In [1]:
import pandas as pd
import json

df_train = pd.read_csv('data/codex/train.txt', sep='\t', header=None, names=['Head','Relation','Tail'])
df_test = pd.read_csv('data/codex/test.txt', sep='\t', header=None, names=['Head','Relation','Tail'])
df_valid = pd.read_csv('data/codex/valid.txt', sep='\t', header=None, names=['Head','Relation','Tail'])

with open('data/codex/entities.json', 'r') as file:
    entities = json.load(file)

with open('data/codex/relations.json', 'r') as file:
    relations = json.load(file)

def id2name_df(id_df:pd.DataFrame, entities_dict:dict, relations_dict:dict)-> pd.DataFrame:
    name_dict = {}
    for id, row in id_df.iterrows():
        # get id
        head_id = row['Head']
        relation_id = row['Relation']
        tail_id = row['Tail']
        # get label out of id
        head = entities_dict[head_id]['label']
        relation = relations_dict[relation_id]['label']
        tail = entities_dict[tail_id]['label']
        name_dict[id] = [head,relation,tail]
    name_df = pd.DataFrame.from_dict(name_dict,orient='index',columns=['Head','Relation','Tail'])
    return name_df

df_train_name = id2name_df(df_train, entities, relations)
df_test_name = id2name_df(df_test, entities, relations)
df_valid_name = id2name_df(df_valid, entities, relations)

df_name = pd.concat([df_train_name, df_test_name, df_valid_name]).reset_index(drop=True)

/var/folders/_3/wtwzgv1d3rlfz233qkf36kg00000gp/T/ipykernel_21366/403170735.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
def compute_missing_df(original_df:pd.DataFrame, sample_df:pd.DataFrame) -> pd.DataFrame:
    df_missing = original_df[~df_name.apply(tuple, axis=1).isin(sample_df.apply(tuple, axis=1))]
    return df_missing

def compute_coverage(filtred_df :pd.DataFrame, df_missing:pd.DataFrame) -> float:
    df_coverage = filtred_df[filtred_df.apply(tuple, axis=1).isin(df_missing.apply(tuple, axis=1))]
    coverage = len(df_coverage) / len(df_missing)
    return coverage

In [7]:
from cand_gen import triple_gen

df_name_sample = df_name.sample(int(0.8*len(df_name)))
candidates_df = triple_gen.generate_all_candidates(df_name_sample)
df_missing = compute_missing_df(df_name, df_name_sample)
coverage = compute_coverage(candidates_df, df_missing)

In [8]:
coverage

0.9833082501026132

In [13]:
coverage_dict = {}
cand_len_dict = {}
sample_size_list = [0.8, 0.75, 0.70, 0.65, 0.6, 0.5, 0.4,0.2]
for sample_size in sample_size_list:
    df_name_sample = df_name.sample(int(sample_size*len(df_name)))
    candidates_df = triple_gen.generate_all_candidates(df_name_sample)
    df_missing = compute_missing_df(df_name, df_name_sample)
    coverage = compute_coverage(candidates_df, df_missing)
    coverage_dict[sample_size] = coverage
    cand_len_dict[sample_size] = len(candidates_df)

In [12]:
coverage_dict

{0.8: 0.9807087152825283,
 0.75: 0.9794220665499125,
 0.7: 0.9808446593085834,
 0.65: 0.9790477679618482,
 0.6: 0.9792037214393214,
 0.5: 0.9718147985989493,
 0.4: 0.9533430630301925,
 0.2: 0.7867966478536002}

In [14]:
cand_len_dict

{0.8: 1245265,
 0.75: 1212345,
 0.7: 1151653,
 0.65: 1101212,
 0.6: 1023309,
 0.5: 860961,
 0.4: 706982,
 0.2: 285467}

In [18]:
cand_len_dict[sample_size]

285467